# Neutrino Physics in qg Space: Simulation and Falsification Audit (Part VII)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_neutrinos_en.ipynb)

This notebook implements the interactive experiments of **neutrino oscillation and falsification audits** presented in *Computación, información y física cuántica en el espacio qang* (35 chapters, Part VII: Experimental Evidence and Falsification Audit).

Neutrinos are produced and detected in flavour states ($e, \mu, \tau$) but propagate as mass eigenstates. In the $qg$ framework, flavour oscillation maps to parametric rotations within a conserved Hamming-weight subspace.

Fully updated to **`qang 0.14.0`** and `qang.integrations`, this notebook executes runs on IBM calibrated noisy backends (`FakeBrisbane`, `FakeTorino`), comparing raw readouts against $qg$ symmetry-filtered sectors.

### Chapter & Research Notes Mapping (Part VII)
| Chapter / § | Physical System | $qg$ Formulation & Metric | Falsification & Error Reduction |
|---|---|---|---|
| Ch. 28 (§151) | Vacuum oscillations (3 flavours, 3 qubits) | $qg_Z = \cos^2(2\theta) + \sin^2(2\theta)\cos\Delta$; $\theta = \frac{1}{4}\arccos(qg_Z)$ at peak | Filter reduces error $\sim 3.5\times$; $\theta$-independent shot noise |
| Ch. 29 (§152) | Earth matter effects (MSW) | Matter-induced CP asymmetry imitation | Matter fakes CP violation ($A_{CP} = 0.042$ at $\delta_{CP}=0$). Filtered circuit extracts true signal |
| Ch. 30 (§154) | Collective oscillations in dense $\nu$ gas | $N_{\nu_2}$ conservation in mass basis ($H_{int} \propto \sum \sigma_i \cdot \sigma_j$) | Weight-conservation holds in mass basis; $2.0\text{--}2.8\times$ noise mitigation |
| Ch. 31 (§156) | Solar adiabatic MSW conversion | $P_{ee}$ strictly linear in $qg_Z$ of $\nu_2$ qubit | Slab circuit ($r=8$) cuts noise; single RBS compilation outperforms Trotter |
| Ch. 32 (§163) | Mass hierarchy, Majorana phases & sterile $\nu$ | Sensitivity to $\text{sign}(\Delta m^2_{31})$ vs gauge phase cancellation | Majorana phases are mathematically undetectable in oscillation circuits |

## 1. Environment Setup (`qang >= 0.14.0`)

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "qiskit-aer", "qiskit-ibm-runtime", "matplotlib"], check=False)
importlib.invalidate_caches()

import qang
from qang import neutrino as nu
from qang import ibm
from qang import formulation as F
from qang import integrations

print(f"Loaded framework: qang v{qang.__version__}")

## 2. Shared Setup on Calibrated Noisy Simulators

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import transpile
from qiskit.quantum_info import Statevector

BACKENDS = ("FakeBrisbane", "FakeTorino")
FR = "filtered_readout"
T0 = time.time()

def get_sampler(backend):
    try:
        from qiskit_ibm_runtime.executor_sampler import Sampler
    except ImportError:
        from qiskit_ibm_runtime import SamplerV2 as Sampler
    return Sampler(mode=backend)

def execute_qg_circuits(circuits, b, weight, shots=4000):
    ts = [transpile(qc, b, optimization_level=1, seed_transpiler=137) for qc in circuits]
    sampler = get_sampler(b)
    res = sampler.run(ts, shots=shots).result()
    return [ibm.ibm_qg(r, t, b, weight=weight) for r, t in zip(res, ts)]

## 3. Vacuum Oscillations and 3-Flavour Weight Preservation (Ch. 28, §151)

In [ ]:
rng = np.random.default_rng(151)
N_SHOTS = 1000
print(f"theta   mean theta_hat   std      theory ({nu.mixing_angle_std(N_SHOTS):.4f})")
for th in (0.1, 0.3, 0.5, 0.7):
    k = rng.binomial(N_SHOTS, nu.two_flavour_probability(th, math.pi), 2000)
    est = np.array([nu.mixing_angle_from_maximum(1 - 2 * x / N_SHOTS) for x in k])
    print(f" {th:4.1f}   {est.mean():.4f}           {est.std(ddof=1):.4f}   {nu.mixing_angle_std(N_SHOTS):.4f}")

L, DCP = 1300.0, -math.pi / 2
ENERGIES = (0.8, 1.2, 1.6, 2.0, 2.6, 3.2, 4.0, 5.0)
exact = np.array([nu.three_flavour_probabilities(L, E, DCP)[1] for E in ENERGIES])

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(ENERGIES, exact[:, 1], "k-", lw=1.8, label="Exact P(mu->mu)")

for name in BACKENDS:
    b = getattr(fp, name)()
    outs = execute_qg_circuits([nu.three_flavour_circuit("mu", L, E, DCP) for E in ENERGIES], b, 1, 2000)
    err = {k: np.abs(np.array([ibm.occupations(o, k) for o in outs]) - exact).mean() for k in ("raw", FR)}
    print(f"{name:<12}: Mean error raw {err['raw']:.3f} | Filtered+Readout {err[FR]:.3f} (Gain {err['raw']/err[FR]:.1f}x)")
    ax.plot(ENERGIES, [ibm.occupations(o, "raw")[1] for o in outs], "x--", alpha=0.7, label=f"{name} raw")
    ax.plot(ENERGIES, [ibm.occupations(o, FR)[1] for o in outs], "o:", lw=1.5, label=f"{name} filtered")

ax.set_xlabel("E (GeV)")
ax.set_ylabel("P(mu -> mu)")
ax.set_title("3-Flavour Oscillation with qg Symmetry Filter")
ax.legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()

## 4. Earth Matter Effects & Faked CP Violation (Ch. 29, §152)

In [ ]:
RHO, R = 2.85, 2
acp_exact = nu.cp_asymmetry(L, 2.6, 0.0, rho=RHO)
sv = lambda anti: np.abs(Statevector(nu.matter_trotter_circuit("mu", L, 2.6, 0.0, RHO, R, antineutrino=anti, measure=False)).data[1]) ** 2
print(f"A_CP (delta_CP = 0): Exact {acp_exact:.4f} | Noiseless Trotter r={R}: {sv(False) - sv(True):.4f}")

for name in BACKENDS:
    b = getattr(fp, name)()
    oc = execute_qg_circuits([nu.matter_trotter_circuit("mu", L, 2.6, 0.0, RHO, R, antineutrino=anti) for anti in (False, True)], b, 1, 10000)
    acp = {k: ibm.occupations(oc[0], k)[0] - ibm.occupations(oc[1], k)[0] for k in ("raw", FR)}
    print(f"{name:<12}: Matter-only A_CP raw {acp['raw']:.4f} | Filtered+Readout {acp[FR]:.4f}")

## 5. Collective Neutrino Oscillations (Ch. 30, §154)

In [ ]:
OMEGAS, MU, THETA, DT = (0.25, 0.5, 0.75, 1.0), 4.0, 0.195, 0.5
TIMES = (0.5, 1.0, 1.5, 2.0)
circ = lambda t, m=True: nu.collective_trotter_circuit(OMEGAS, MU, THETA, "0011", t, int(round(t / DT)), "mass", "mass", measure=m)
idx = np.arange(16)
ref = []
for t in TIMES:
    p = Statevector(circ(t, False)).probabilities()
    ref.append([p[(idx >> i) & 1 == 1].sum() for i in range(4)])
ref = np.array(ref)

for name in BACKENDS:
    b = getattr(fp, name)()
    outs = execute_qg_circuits([circ(t) for t in TIMES], b, 2, 4000)
    err = {k: np.abs(np.array([ibm.occupations(o, k) for o in outs]) - ref).mean() for k in ("raw", FR)}
    print(f"{name:<12}: Mean |P(nu_2) error| raw {err['raw']:.3f} | Filtered+Readout {err[FR]:.3f} (Gain {err['raw']/err[FR]:.1f}x)")

## 6. Solar Adiabatic MSW Transition & Falsification Audit (Ch. 31, §156)

In [ ]:
H, X, RS = 100.0, 800.0, 8
E_SOL = (0.0005, 0.001, 0.002, 0.004, 0.006, 0.010, 0.015)
pee = lambda p2s: np.array([nu.solar_pee_from_p2(float(p)) for p in p2s])
ex = pee([nu.solar_p2_exact(E, H, X) for E in E_SOL])
slab = pee([nu.solar_p2(E, H, X, RS) for E in E_SOL])

for name in BACKENDS:
    b = getattr(fp, name)()
    so = execute_qg_circuits([nu.solar_circuit(E, H, X, RS) for E in E_SOL], b, 1, 4000)
    co = execute_qg_circuits([nu.solar_circuit(E, H, X, RS, compiled=True) for E in E_SOL], b, 1, 4000)
    est_slab_f = pee([ibm.occupations(o, FR)[1] for o in so])
    est_comp_f = pee([ibm.occupations(o, FR)[1] for o in co])
    print(f"{name:<12} | Slab r=8 error: {np.abs(est_slab_f - slab).mean():.4f} | Compiled error: {np.abs(est_comp_f - slab).mean():.4f}")

## 7. Falsification Boundaries: Majorana Invariance & Sterile Modes (Ch. 32, §163)

In [ ]:
L163, E163, RHO163 = 1300.0, 2.6, 2.85
inv = nu.majorana_invisible(L163, E163, -1.57, 1.0, 2.5, rho=RHO163)
print(f"Majorana phase sensitivity in oscillation probability: {inv:.2e} (Strictly zero)")
b = getattr(fp, BACKENDS[0])()
m = [transpile(nu.majorana_circuit("mu", L163, E163, -1.57, a, c), b, optimization_level=1, seed_transpiler=137)
     for a, c in ((0, 0), (math.pi / 2, 3 * math.pi / 4))]
print(f"Transpiled circuits with/without Majorana phases are identical: {m[0] == m[1]}")

## Conclusions
1. **Symmetry Filter Gain:** The weight-preserving sector verification in $qg$ metric cuts NISQ device error by $2\times$ to $5\times$ without requiring ancillary qubits.
2. **Audit & Scope:** Demonstrates that oscillation tests cannot identify Majorana phases, while single RBS compilations expose the classical nature of decohered solar populations.